## This is the starting dataset for the infleqtion project. 
---
The idea here is to create the circuits first using cirq to make sure they work as expected, then three json entries would be made per circuit noting the prompting style that should be used to create the desired circuit. 

#### Prompting Styles 
- [[Technical, using precise quantum notation of desired circuit, ex. "Prepare |phi+> on two qubits" ],
    [Conversational, using everday language, ex. "Entangle 2 qubits" ],
    [Narrtive, describes the outcome of a circuit, ex. "Generate a circuit that produce 00 and 11 with equal probabilities]]

In [3]:
#bell state circuit
import cirq

bell_circuit = cirq.Circuit()
q0, q1 = cirq.LineQubit.range(2)
bell_circuit.append(cirq.H(q0)) 
bell_circuit.append(cirq.CNOT(q0, q1)) 
bell_circuit
print(cirq.final_state_vector(bell_circuit))


[0.70710677+0.j 0.        +0.j 0.        +0.j 0.70710677+0.j]


In [4]:
h_circuit = cirq.Circuit()

q0 = cirq.LineQubit.range(1)
h_circuit.append(cirq.H(q0[0]))
print(h_circuit)


0: ───H───


In [5]:

#Syntactic Check -- Makes sure that everything actually runs 
def check_syntax(llm_code: str) -> dict:
    try:
        exec_globals = {}
        exec(llm_code, exec_globals)
        return {"passed": True, "error": None}
    except Exception as e:
        return {"passed": False, "error": str(e)}

In [6]:
#Structural Check -- does it have the right gates and qubits
def check_structure(llm_circuit: cirq.Circuit, control_circuit: cirq.Circuit,) -> dict:
    results = {} 

    # Check amount of qubits 
    llm_qubits = len(llm_circuit.all_qubits())
    control_qubits = len(control_circuit.all_qubits())
    results["qubit_count_match"] = llm_qubits == control_qubits 

    # Check gate type present 
    def get_gates(circuit):
        return set(
            type(op.gate).__name__ #Collects names of gates as a set
            for moment in circuit
            for op in moment.operations
            if op.gate is not None
        )

    llm_gates = get_gates(llm_circuit)
    control_gates = get_gates(control_circuit)
    results["gate_types_match"] = control_gates.issubset(llm_gates)
    results["llm_gates"] = list(llm_gates) 
    results["control_gates"] = list(control_gates)

    #Measurement Check

    llm_has_measurement = any(

        isinstance(op.gate, cirq.MeasurementGate)

        for moment in llm_circuit
        for op in moment.operations
    )

    control_has_measurement = any(
        isinstance(op.gate, cirq.MeasurementGate)
        for moment in control_circuit
        for op in moment.operations
    )
    results["measurement_match"] = llm_has_measurement == control_has_measurement

    results["circuit_depth_llm"] = len(llm_circuit)
    results["circuit_depth_control"] = len(control_circuit)
    results["depth_within_tolerance"] = len(llm_circuit) <= len(control_circuit) * 2 

    return results

In [70]:
# Semantic Check - checks the output distribution
import numpy as np
def check_semantics(llm_circuit: cirq.Circuit, control_circuit: cirq.Circuit, reps: int = 1000, tolerance: float = 0.1) -> dict:
    simulator = cirq.Simulator()
    
    #MESUREMENT FUNCTION '
    def ensure_measured(circuit):
        qubits = sorted(circuit.all_qubits())
        if not any(isinstance(op.gate, cirq.MeasurementGate)
                   for moment in circuit for op in moment.operations):
            circuit = circuit + cirq.measure(*qubits, key='result') #* unpacks list of qubits as their own arguments 
            #key+'result' makes a name for measurements to call

        return circuit

    llm_measured = ensure_measured(llm_circuit.copy())  
    control_measured = ensure_measured(control_circuit.copy())
    
    llm_result = simulator.run(llm_measured, repetitions=reps)
    control_result = simulator.run(control_measured, repetitions=reps)

    #Build Probability(Result) distributions
    def get_distribution(result, reps):
        counts = result.multi_measurement_histogram(keys=['result']) # counts the number of outcomes 
        return {k: v/reps for k, v in counts.items()}  #turns number into probability
    llm_dist = get_distribution(llm_result, reps)
    control_dist = get_distribution(control_result, reps)

    #Compare distributions 
    all_keys = set(llm_dist.keys()) | set(control_dist.keys())
    
    distribution_match = all(
        abs(llm_dist.get(k,0) - control_dist.get(k,0)) <= tolerance
        for k in all_keys 
    )

    # STATE VECTOR
    def get_state_vector(circuit):
        no_measurement = cirq.drop_terminal_measurements(circuit)
        return cirq.final_state_vector(no_measurement)

    llm_vector = get_state_vector(llm_circuit)
    control_vector = get_state_vector(control_circuit)

    #comparing the vectors
    fidelity = float(abs(np.dot(np.conj(llm_vector), control_vector))**2)
    assert cirq.equal_up_to_global_phase(llm_vector, control_vector)
    # np.dot(np.conj()creates inner products of the two vectors 
    # the absolute value and square on it makes the quantum fidelity 
    # quantum fidelity of 1 means that the two vectors are the same and 0 means that they are orthagonal or completely different. 
    
    # RETURN combined result
    return {
        "passed": distribution_match, 
        "passed": fidelity >= 0.9,
        "fidelity_score": fidelity,
        "llm_distribution": {str(k): v for k, v in llm_dist.items()},
        "control_distribution": {str(k): v for k, v in control_dist.items()}
    }

    
   



    

       

In [8]:
# THE BIG ONE 

def validate_circuit(llm_code: str, control_circuit: cirq.Circuit) -> dict:
    report = {}
    # Step 1: Syntax
    syntax = check_syntax(llm_code)
    report["syntax"] = syntax

    if not syntax["passed"]:
        report["verdict"] = "invalid_code"
        return report
    
    #Step 2: Extracting da circuit
    exec_globals = {}
    exec(llm_code, exec_globals)
    llm_circuit = next(
        (v for v in exec_globals.values() if isinstance(v, cirq.Circuit)),
        None
    )
    if llm_circuit is None:
        report["verdict"] = "no_circuit_found"
        return report
    
    # Step 3: Structure
    structure = check_structure(llm_circuit, control_circuit)
    report["structure"] = structure

    # Step 4: Semantics
    semantics = check_semantics(llm_circuit, control_circuit)
    report["semantics"] = semantics 

    #Step 5: FINAL Outcome 
    if not structure["qubit_count_match"]:
        report["verdict"] = "missing_qubits"
    elif not structure["gate_types_match"]:
        report["verdict"] = "wrong_gate"
    elif not structure["measurement_match"]:
        report["verdict"] = "missing_measurement"
    elif not semantics["passed"]:
        report["verdict"] = "semantic_fail"
    else:
        report["verdict"] = "success"

    return report


In [9]:
bell_test_code = """import cirq

# Create a quantum circuit with 2 qubits\nqubits = cirq.LineQubit.range(2)
circuit = cirq.Circuit()
circuit.append([
    cirq.H(qubits[0]),
    cirq.CNOT(qubits[0], qubits[1])
])
"""
validate_circuit(bell_test_code, bell_circuit)

{'syntax': {'passed': True, 'error': None},
 'structure': {'qubit_count_match': True,
  'gate_types_match': True,
  'llm_gates': ['CXPowGate', 'HPowGate'],
  'control_gates': ['CXPowGate', 'HPowGate'],
  'measurement_match': True,
  'circuit_depth_llm': 2,
  'circuit_depth_control': 2,
  'depth_within_tolerance': True},
 'semantics': {'passed': True,
  'fidelity_score': 0.9999998807907104,
  'llm_distribution': {'(0,)': 0.495, '(3,)': 0.505},
  'control_distribution': {'(0,)': 0.516, '(3,)': 0.484}},
 'verdict': 'success'}

In [ ]:
bell_test2_code = """import cirq\n\n# Create a quantum circuit with 2 qubits\nqubits = cirq.LineQubit.range(2)  # qubits 0 and 1\ncircuit = cirq.Circuit()\n\n# Add gates to create the Bell state |Φ⁺⟩ = (|00⟩ + |11⟩)/√2\ncircuit.append([\n    # Apply Hadamard gate to the first qubit\n    cirq.H(qubits[0]),\n\n    # Apply CNOT gate with first qubit as control and second as target\n    cirq.CNOT(qubits[0], qubits[1])\n])\n\n# Print the circuit\nprint("Circuit to create Bell state |Φ⁺⟩:")\nprint(circuit)\n\n# Visualize the circuit\nprint("\\nCircuit diagram:")\nprint(circuit.to_text_diagram())\n"""

validate_circuit(bell_test2_code, bell_circuit)

In [ ]:
#JSON Database for Notion Work
{
  "database_name": "Quantum Circuit Tracker",
  "record_count": 20,
  "schema": {
    "Circuit": "object",
    "Description of Utility": "object",
    "Difficulty": "object",
    "Expected Gates": "object",
    "Status": "object"
  },
  "records": [
    {
      "Circuit": "Two-Layer Variational Circuit",
      "Description of Utility": "Parameterized ansatz with two layers of rotations and entanglers on three qubits. Used in VQE and quantum ML.",
      "Difficulty": "Advanced",
      "Expected Gates": "RY, CNOT, RY, CNOT",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Phase Kickback",
      "Description of Utility": "Demonstrates how phase from a target qubit transfers back to a control qubit. Core subroutine in Shor's and Grover's algorithms.",
      "Difficulty": "Advanced",
      "Expected Gates": "H, CNOT, H",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Bell State (|Φ+⟩)",
      "Description of Utility": "Creates a maximally entangled two-qubit state producing only |00⟩ and |11⟩ outcomes. Core benchmark for this project.",
      "Difficulty": "Beginner",
      "Expected Gates": "H, CNOT",
      "Status": "Created Base Circuit"
    },
    {
      "Circuit": "Hadamard (Single Qubit)",
      "Description of Utility": "Puts a single qubit into equal superposition of |0⟩ and |1⟩. Foundational gate used in nearly every quantum algorithm.",
      "Difficulty": "Beginner",
      "Expected Gates": "H",
      "Status": "Learned Function"
    },
    {
      "Circuit": "RZ Rotation",
      "Description of Utility": "Rotates a qubit around the Z-axis. Used for phase adjustments in variational circuits.",
      "Difficulty": "Intermediate",
      "Expected Gates": "RZ(theta)",
      "Status": "Learned Function"
    },
    {
      "Circuit": "CNOT Gate",
      "Description of Utility": "Flips target qubit conditional on control qubit being |1⟩. Primary two-qubit entangling gate in most frameworks.",
      "Difficulty": "Beginner",
      "Expected Gates": "CNOT",
      "Status": "Learned Function"
    },
    {
      "Circuit": "Grover's Diffusion Operator",
      "Description of Utility": "Amplitude amplification step of Grover's search algorithm. Reflects amplitudes about their mean to boost target state probability. Advanced stretch circuit.",
      "Difficulty": "Expert",
      "Expected Gates": "H, X, H, CNOT, H, X, H",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Ambiguous Entanglement Prompt",
      "Description of Utility": "Intentionally vague prompt like 'Make two qubits the same somehow.' Tests whether LLM asks for clarification or picks a valid interpretation.",
      "Difficulty": "Intermediate",
      "Expected Gates": "TBD — model chooses",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Quantum Teleportation",
      "Description of Utility": "Transfers an unknown qubit state using entanglement and classical communication. Three-qubit protocol beyond project scope — stretch goal.",
      "Difficulty": "Expert",
      "Expected Gates": "H, CNOT, CNOT, H, M, X, Z",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Toffoli Gate (CCNOT)",
      "Description of Utility": "Three-qubit gate — flips target only if both control qubits are |1⟩. Universal for classical reversible computation.",
      "Difficulty": "Advanced",
      "Expected Gates": "CCNOT",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Pauli-Z Gate",
      "Description of Utility": "Applies a phase flip — leaves |0⟩ unchanged and flips the sign of |1⟩. Used in phase kickback and error correction.",
      "Difficulty": "Beginner",
      "Expected Gates": "Z",
      "Status": "Learned Function"
    },
    {
      "Circuit": "Bell State with Measurement",
      "Description of Utility": "Bell state with measurement operations added to both qubits. Tests whether LLM correctly appends measurements when prompted.",
      "Difficulty": "Beginner",
      "Expected Gates": "H, CNOT, M",
      "Status": "Created Base Circuit"
    },
    {
      "Circuit": "RY Rotation",
      "Description of Utility": "Rotates a qubit around the Y-axis by a specified angle. Common in ansatz layers for quantum ML.",
      "Difficulty": "Intermediate",
      "Expected Gates": "RY(theta)",
      "Status": "Learned Function"
    },
    {
      "Circuit": "GHZ State (3-qubit)",
      "Description of Utility": "Entangles three qubits into a superposition of |000⟩ and |111⟩. Extends Bell state logic to three qubits.",
      "Difficulty": "Intermediate",
      "Expected Gates": "H, CNOT, CNOT",
      "Status": "Created Base Circuit"
    },
    {
      "Circuit": "GHZ State with Measurement",
      "Description of Utility": "GHZ circuit with measurements on all three qubits. Semantic check: outcomes should be only 000 and 111.",
      "Difficulty": "Intermediate",
      "Expected Gates": "H, CNOT, CNOT, M",
      "Status": "Learned Function"
    },
    {
      "Circuit": "SWAP Gate",
      "Description of Utility": "Exchanges the states of two qubits. Used in routing algorithms and circuit compilation.",
      "Difficulty": "Intermediate",
      "Expected Gates": "SWAP",
      "Status": "Haven't Touched"
    },
    {
      "Circuit": "Single-Qubit Measurement",
      "Description of Utility": "Puts qubit in superposition then measures. Should yield ~50/50 outcomes. Tests measurement-only prompts.",
      "Difficulty": "Beginner",
      "Expected Gates": "H, M",
      "Status": "Learned Function"
    },
    {
      "Circuit": "RX Rotation",
      "Description of Utility": "Rotates a qubit around the X-axis by a specified angle. Used in variational and parameterized circuits.",
      "Difficulty": "Intermediate",
      "Expected Gates": "RX(theta)",
      "Status": "Learned Function"
    },
    {
      "Circuit": "Pauli-X Gate",
      "Description of Utility": "Flips a qubit from |0⟩ to |1⟩ or vice versa. Quantum equivalent of a classical NOT gate.",
      "Difficulty": "Beginner",
      "Expected Gates": "X",
      "Status": "Learned Function"
    },
    {
      "Circuit": "Quantum Fourier Transform (3-qubit)",
      "Description of Utility": "Computes the discrete Fourier transform on quantum amplitudes. Subroutine in Shor's algorithm and phase estimation. More complex than project scope.",
      "Difficulty": "Expert",
      "Expected Gates": "H, CPHASE, SWAP",
      "Status": "Haven't Touched"
    }
  ]
}

In [ ]:
import json
[{
"id": "bell_technical_001",
"prompt": "Prepare the Bell state |Phi+> on two qubits as a Cirq circuit",
"prompt_style":"technical",
"category": "bell_state",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H", "CNOT"],
"expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability."
"resulted code": "```python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit(\n    # Apply Hadamard to qubit0\n    cirq.H(qubit0),\n    # Apply CNOT with qubit0 as control and qubit1 as target\n    cirq.CNOT(qubit0, qubit1)\n)\n\n# Print the circuit\nprint("Circuit to prepare |Φ⁺⟩:")\nprint(circuit)\n```",
"validated verdict": "success"
},
{
"id": "bell_conversational_001",
"prompt": "Entangle 2 qubits as a Cirq Circuit",
"prompt_style":"conversational",
"category": "bell_state",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H", "CNOT"],
"expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
"resulted_code": "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n",
"validated_verdict:": "invalid_code, inccorectly structured print statement, hallucinated state vector code",
},
{
"id": "bell_narrative_001",
"prompt": "Generate a quantum circuit that produces 00 and 11 with equal probibility",
"prompt_style":"narrative",
"category": "bell_state",
"difficulty": "easy",
"num_qubits": 2,
"expected_gates": ["H", "CNOT"],
"expected_behavior": "Measurement outcomes 00 and 11 should dominate with roughly equal probability.",
"resulted_code": "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\n print(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n",
"validated_verdict": "invalid code, spacing error", 

}
{
"id": "h_technical_001",
"prompt": "Apply a Hadamard gate to a single qubit initialized in the |0⟩ computational basis state and return the resulting Cirq circuit.",
"prompt_style":"technical",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition"
"resulted code:```python\nimport cirq\n\n# Create a single qubit circuit\nqubit = cirq.LineQubit(0)\ncircuit = cirq.Circuit()\n\n# Add the Hadamard gate to the qubit\ncircuit.append(cirq.H(qubit))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n```",
"validated verdict: success"
""
},
{
"id": "h_conversational_001",
"prompt": "Put a qubit in superposition and model as a Cirq Circuit",
"prompt_style":"conversational",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n",
"validated_verdict:": "invalid_code",
},
{
"id": "h_narrative_001",
"prompt": "Generate a Cirq quantum circuit that produces 0 and 1 with equal probibility",
"prompt_style":"narrative",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n```",
"validated_verdict": "success", }
{
"id": "tele_narrative_001",
"prompt": "Generate a cirq circuit modeling quantum teleportation.",
"prompt_style":"narrative",
"category": "h_state",
"difficulty": "easy",
"num_qubits": 1,
"expected_gates": ["H"],
"expected_behavior": "After H is applied the qubit is put into superposition",
"resulted_code": "```python\nimport cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n```",
"validated_verdict": "success", }
]


SyntaxError: invalid syntax (1161551386.py, line 11)

In [97]:
teleportion_circuit = cirq.Circuit()

q0, q1,q2 = cirq.LineQubit.range(3)
teleportion_circuit += cirq.H(q2)
teleportion_circuit += cirq.CX(q2, q1)
teleportion_circuit += cirq.CX(q0, q1)
teleportion_circuit += cirq.H(q0)
teleportion_circuit += cirq.measure(q0, key='q0')
teleportion_circuit += cirq.measure(q1, key='q1')
teleportion_circuit += cirq.Z(q2).with_classical_controls('q0')
teleportion_circuit += cirq.X(q2).with_classical_controls('q1')
print(teleportion_circuit)

0: ────────────@───H───M───────────
               │       ║
1: ────────X───X───M───╫───────────
           │       ║   ║
2: ────H───@───────╫───╫───Z───X───
                   ║   ║   ║   ║
q0: ═══════════════╬═══@═══^═══╬═══
                   ║           ║
q1: ═══════════════@═══════════^═══


In [98]:
teleportation_test = """import cirq\n\n# Create a qubit for the state to be teleported\nqubit_for_teleportation = cirq.LineQubit(0)\n\n# Create a qubit for the entangled pair\nqubit_entangled_pair = cirq.LineQubit(1)\n\n# Create a qubit for the receiver\nqubit_receiver = cirq.LineQubit(2)\n\n# Create the entangled pair (EPR pair)\nepr_pair = cirq.H(qubit_entangled_pair)\nepr_pair_with_control = cirq.X(qubit_entangled_pair) ** cirq.ControlledBy(cirq.X(qubit_receiver))\n\n# Prepare the state to be teleported\nstate_to_teleport = cirq.X(qubit_for_teleportation)\n\n# Apply the teleportation circuit\nteleportation_circuit = cirq.Circuit(\n    epr_pair,\n    cirq.CNOT(qubit_for_teleportation, qubit_entangled_pair),\n    cirq.H(qubit_for_teleportation),\n    cirq.measure(qubit_for_teleportation, key='teleported_state'),\n    cirq.measure(qubit_entangled_pair, key='entangled_pair'),\n    cirq.measure(qubit_receiver, key='receiver_state')\n)\n\n# Apply the measurement on the receiver's qubit\nreceiver_measurement = cirq.measure(qubit_receiver, key='receiver_measurement')\n\n# Apply the correction to the receiver's qubit\ncorrection = cirq.X(qubit_receiver) ** cirq.ControlledBy(cirq.X(qubit_entangled_pair))\n\n# Combine all the operations into a single circuit\nteleportation_circuit = cirq.Circuit(\n    teleportation_circuit,\n    receiver_measurement,\n    correction\n)\n\n# Run the circuit\nsimulator = cirq.Simulator()\nresult = simulator.run(teleportation_circuit, repetitions=1000)\n\n# Print the results\nprint(result)\n"""
validate_circuit(teleportation_test, teleportion_circuit)

{'syntax': {'passed': False,
  'error': "module 'cirq' has no attribute 'ControlledBy'"},
 'verdict': 'invalid_code'}

In [96]:
test_circuit = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("\\nFinal state:)\nprint(result.final_state)\n"""

validate_circuit(test_circuit, h_circuit)

{'syntax': {'passed': False,
  'error': 'unterminated string literal (detected at line 23) (<string>, line 23)'},
 'verdict': 'invalid_code'}

In [ ]:
bt_code = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\n print(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n"""
bell_test1_code = """import cirq

# Define the qubits
qubit0 = cirq.LineQubit(0)
qubit1 = cirq.LineQubit(1)

# Create the circuit
circuit = cirq.Circuit()
circuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0
circuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target

# Print the circuit
print("Circuit:")
print(circuit)

# Simulate the circuit
simulator = cirq.Simulator()
result = simulator.simulate(circuit)
final_state = result.final_state_vector

print("\\nFinal state vector:")
print(final_state)

# Measure the probabilities
print("\\nMeasurement probabilities:")
print(result.measurements)
"""
validate_circuit(bt_code, bell_circuit)


In [ ]:
bell_circuit8 = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create the circuit\ncircuit = cirq.Circuit()\ncircuit.append(cirq.H(qubit0))  # Apply Hadamard to qubit0\ncircuit.append(cirq.CNOT(qubit0, qubit1))  # Apply CNOT with qubit0 as control and qubit1 as target\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nfinal_state = result.final_state_vector\n\nprint("\\nFinal state vector:")\nprint(final_state)\n\n# Measure the probabilities\nprint("\\nMeasurement probabilities:")\nprint(result.measurements)\n"""
import cirq
qubit0 = cirq.LineQubit(0)
qubit1 = cirq.LineQubit(1)

circuit = cirq.Circuit()
# Add a Hadamard gate to qubit0 to create superposition
circuit.append(cirq.H(qubit0))
# Add a CNOT gate to entangle qubit0 and qubit1
circuit.append(cirq.CNOT(qubit0, qubit1))
# Print the circuit
print("Circuit:")
print(circuit)


# Simulate the circuit
simulator = cirq.Simulator()
result = simulator.simulate(circuit)
print("Final state:")
print(result.final_state_vector)
validate_circuit(bell_circuit8, bell_circuit)


In [ ]:
bell_test4_code = """import cirq\n\n# Define the qubits\nqubit0 = cirq.LineQubit(0)\nqubit1 = cirq.LineQubit(1)\n\n# Create a circuit\ncircuit = cirq.Circuit()\n\n# Add a Hadamard gate to qubit0 to create superposition\ncircuit.append(cirq.H(qubit0))\n\n# Add a CNOT gate to entangle qubit0 and qubit1\ncircuit.append(cirq.CNOT(qubit0, qubit1))\n\n# Print the circuit\nprint("Circuit:")\nprint(circuit)\n\n# Simulate the circuit\nsimulator = cirq.Simulator()\nresult = simulator.simulate(circuit)\nprint("Final state:")\nprint(result.final_state_vector)\n"""

validate_circuit(bell_test4_code, bell_circuit)